# Demo: Rainfall Occurrence Prediction from Radiosonde Data

This notebook demonstrates the core modelling workflow on **sample data** (real sounding indices from WMO 48565 Phuket, with synthetic rainfall labels).  
No TMD rainfall data is required to run this demo.

**Pipeline steps demonstrated:**
1. Feature preparation (F1 thermodynamic set)
2. Year-blocked 5-fold GroupKFold CV — prevents temporal leakage
3. Chronological holdout test (2020–2021)
4. Calibration: Brier score, PR-AUC, reliability diagram
5. SHAP physical plausibility diagnostic

**CV-selected model:** Logistic Regression + F1  
**SHAP comparator:** XGBoost + F3

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, brier_score_loss,
                             average_precision_score, roc_curve)
from sklearn.calibration import calibration_curve
import xgboost as xgb

REPO = Path('..') 
DATA = REPO / 'data' / 'sample' / 'sample_dataset.csv'

TRAIN_YEARS = list(range(2011, 2020))
TEST_YEARS  = [2020, 2021]
RAIN_THRESH = 0.1
RANDOM_SEED = 42
STATION     = 'phuket'
RAIN_COL    = 'RF_phuket'

print('Setup complete.')

## 1. Load and Inspect Data

In [ ]:
df = pd.read_csv(DATA, parse_dates=['date'])
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['y']     = (df[RAIN_COL] >= RAIN_THRESH).astype(int)

print(f'Dataset: {len(df)} rows, {df.year.min()}–{df.year.max()}')
print(f'Overall rain rate: {df.y.mean()*100:.1f}%')
df[['date','PWAT','KINX','CAPE','WS850','WDcos850','RF_phuket','y']].head()

In [ ]:
# Feature sets
F1 = ['SHOW','LIFT','SWET','KINX','VTOT','CAPE','CINS','CINV',
       'EQLV','EQTV','LFCT','LFCV','BRCH','LCLT','LCLP','MLTH','THTK','PWAT']
F3 = F1 + ['WS850','WDsin850','WDcos850','WS700','WS200','VWS','VWS_lo']

train_df = df[df.year.isin(TRAIN_YEARS)].reset_index(drop=True)
test_df  = df[df.year.isin(TEST_YEARS)].reset_index(drop=True)

rr_tr = train_df.y.mean()*100
rr_te = test_df.y.mean()*100
print(f'Train 2011–2019: n={len(train_df)}, rain rate={rr_tr:.1f}%')
print(f'Test  2020–2021: n={len(test_df)},  rain rate={rr_te:.1f}%')
print(f'Prevalence shift: ~{abs(rr_tr-rr_te):.0f} pp')

## 2. Year-Blocked Cross-Validation (Tier 1)

GroupKFold by calendar year — no year appears in both train and validation subsets within a fold. This prevents temporal leakage.

In [ ]:
F1_avail = [c for c in F1 if c in df.columns]
X_tr  = train_df[F1_avail].fillna(train_df[F1_avail].mean()).values
y_tr  = train_df.y.values

gkf = GroupKFold(n_splits=5)
fold_aucs = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_tr, y_tr, groups=train_df.year.values)):
    Xtr, Xva = X_tr[tr_idx], X_tr[va_idx]
    ytr, yva = y_tr[tr_idx], y_tr[va_idx]
    
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr); Xva = sc.transform(Xva)
    
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight='balanced')
    clf.fit(Xtr, ytr)
    proba = clf.predict_proba(Xva)[:,1]
    auc = roc_auc_score(yva, proba)
    fold_aucs.append(auc)
    print(f'  Fold {fold+1}: AUC={auc:.4f}  (held-out years: {sorted(train_df.year.values[va_idx][:3])}...)')

print(f'\nYear-blocked CV AUC: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}')

## 3. Chronological Holdout Test (Tier 2)

In [ ]:
X_te = test_df[F1_avail].fillna(train_df[F1_avail].mean()).values
y_te = test_df.y.values

sc   = StandardScaler()
Xtr2 = sc.fit_transform(X_tr)
Xte2 = sc.transform(X_te)

lr   = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight='balanced')
lr.fit(Xtr2, y_tr)
proba_lr = lr.predict_proba(Xte2)[:,1]

# XGB+F3 comparator
F3_avail = [c for c in F3 if c in df.columns]
X_tr3 = train_df[F3_avail].fillna(train_df[F3_avail].mean()).values
X_te3 = test_df[F3_avail].fillna(train_df[F3_avail].mean()).values
n_neg, n_pos = (y_tr==0).sum(), (y_tr==1).sum()
xgb_clf = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric='logloss', random_state=RANDOM_SEED,
                              verbosity=0, scale_pos_weight=n_neg/n_pos)
xgb_clf.fit(X_tr3, y_tr)
proba_xgb = xgb_clf.predict_proba(X_te3)[:,1]

print(f'LR+F1  holdout AUC: {roc_auc_score(y_te, proba_lr):.4f}')
print(f'XGB+F3 holdout AUC: {roc_auc_score(y_te, proba_xgb):.4f}')

## 4. Calibration Assessment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (lbl, proba_) in zip(axes, [('LR+F1', proba_lr), ('XGB+F3', proba_xgb)]):
    frac, mean_p = calibration_curve(y_te, proba_, n_bins=8, strategy='uniform')
    bs  = brier_score_loss(y_te, proba_)
    pr  = average_precision_score(y_te, proba_)
    roc = roc_auc_score(y_te, proba_)
    
    ax.plot([0,1],[0,1],'k--',lw=1.2,label='Perfect calibration')
    ax.plot(mean_p, frac, 'o-', lw=2, markersize=7, label=lbl)
    ax.set_title(f'{lbl}\nBrier={bs:.3f}  PR-AUC={pr:.3f}  ROC={roc:.3f}', fontsize=10)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Reliability Diagrams — Phuket Holdout 2020-2021\n(sample data with synthetic labels)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()
print('Note: these results use SYNTHETIC labels for demonstration only.')

## 5. SHAP Physical Plausibility Diagnostic

SHAP is applied to XGB+F3 to audit whether learned feature associations are physically plausible (not as evidence that XGB+F3 is the best model).

In [ ]:
explainer = shap.TreeExplainer(xgb_clf)
shap_vals  = explainer.shap_values(X_te3)
mean_abs   = np.abs(shap_vals).mean(axis=0)
top_idx    = np.argsort(mean_abs)[::-1][:10]

print('Top 10 features by mean |SHAP|:')
for i in top_idx:
    print(f'  {F3_avail[i]:12s}  {mean_abs[i]:.4f}')

In [ ]:
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_vals, X_te3, feature_names=F3_avail, show=False, plot_type='dot')
plt.title('SHAP Beeswarm — XGB+F3 (sample data demo)', fontsize=10)
plt.tight_layout()
plt.show()

print('\nExpected top features in real data: PWAT (precipitable water), WDcos850 (low-level wind direction)')
print('These are physically consistent with tropical convective rainfall mechanisms.')

---
## Summary

This demo shows the complete workflow on sample data. To replicate the full paper results:

1. Obtain TMD daily rainfall data and place in `data/raw/rainfall_tmd.csv`
2. Set `USE_SAMPLE = False` in `02_features.py`
3. Run scripts in order: `01_preprocess.py` → `02_features.py` → `03_train.py` → `04_validate.py` → `05_shap.py`

See `PIPELINE.md` for full instructions and expected runtimes.